# Python File Handling Exercises: 34 Coding Problems with Solutions

A practice notebook on reading, writing, appending, and manipulating files — text analysis, binary I/O, directory operations, and chunked processing — each with a concept note, a hint, a solution, and an explanation.

*Adapted for practice from the exercise list at [PYnative](https://pynative.com/python-file-handling-exercises/). Each code cell creates its own sample input file(s) first, so the notebook runs standalone from top to bottom.*

---

## Concepts you'll need

This set covers Python **file I/O**: opening, reading, writing, and manipulating files and directories.

- **Open modes** — `"r"` (read, default, error if missing), `"w"` (write, creates or **truncates**), `"a"` (append, never erases existing content), plus `"b"` suffix for binary (`"rb"`, `"wb"`) which reads/writes raw bytes with no text decoding or newline translation — essential for images, PDFs, and any non-text format.
- **Always use `with open(...) as f:`** — it guarantees the file is closed even if an error occurs partway through. You can open two files in one `with` statement: `with open(a) as fa, open(b) as fb:`.
- **Reading strategies** — `f.read()` loads the whole file as one string (fine for small files); `for line in f:` iterates lazily, one line at a time (memory-efficient for large files); `f.readlines()` loads every line into a list at once (needed when you must look at the end, or index by line number).
- **Writing strategies** — `f.write(string)` writes exactly what you give it, no automatic newlines; `f.writelines(list_of_strings)` writes each string in sequence with no separator added, so each string needs its own trailing `\n` if you want line breaks.
- **The `os` module** — `os.path.exists()`, `os.path.getsize()`, `os.path.isfile()`, `os.listdir()`, `os.rename()`, `os.remove()`, `os.path.join()` (builds paths with the correct separator for the OS) for filesystem-level operations beyond just reading/writing content.
- **Defensive file handling** — check `os.path.exists()` before an operation, or wrap it in `try`/`except FileNotFoundError`, to avoid crashing when a file is missing.
- **Chunked processing** — for very large files (or any binary file), reading a fixed-size chunk in a loop (`while True: chunk = f.read(4096); if not chunk: break`) keeps memory usage constant regardless of file size.

Each exercise below gives a problem, a hint, a solution, and an explanation. Every code cell creates whatever sample file(s) it needs first, so the whole notebook runs top to bottom with no external setup required.

**Note on Exercise 26 and 30:** these operate on a whole directory — for this notebook they use a small sandboxed subfolder so they don't touch any of your other files.

## Exercise 1. Write User Name to File

**Concept:** open() in write mode ('w')

**Problem:** Write a name to a file called user.txt.

**Given:**
```
name = "Alice" (simulating console input)
```

**Expected Output:**
```
Name written to user.txt
```

**Hint:** 'w' mode creates the file if missing, or overwrites it if it exists.

In [ ]:
name = "Alice"  # simulates input("Enter your name: ")

with open("user.txt", "w") as f:
    f.write(name)

print("Name written to user.txt")

with open("user.txt") as f:
    print("Contents of user.txt:", repr(f.read()))

**Explanation:** open("user.txt", "w") opens (or creates) the file in write mode, overwriting any existing content. The with statement guarantees the file closes automatically once the block ends, even if an error occurred inside it. f.write(name) writes the string exactly as given — no newline is added automatically.

## Exercise 2. Read and Print Complete File

**Concept:** f.read() to load the whole file as one string

**Problem:** Open a file and print its entire contents at once.

**Given:**
```
data.txt containing "Hello, World!"
```

**Expected Output:**
```
Hello, World!
```

**Hint:** 'r' (read mode, the default) raises FileNotFoundError if the file doesn't exist.

In [ ]:
with open("data.txt", "w") as f:
    f.write("Hello, World!")

with open("data.txt", "r") as f:
    content = f.read()

print(content)

**Explanation:** f.read() loads the entire file into one string in a single call, including any newline characters — suitable for small files where memory isn't a concern. Printing that string reproduces the file's original layout exactly.

## Exercise 3. Read File Line by Line Using Loop

**Concept:** iterating a file object directly, one line at a time

**Problem:** Print each line of a file using a loop, without loading the whole file into memory at once.

**Given:**
```
lines.txt with 3 lines
```

**Expected Output:**
```
Line 1
Line 2
Line 3
```

**Hint:** `for line in f:` is memory-efficient since Python reads lazily, one line per iteration.

In [ ]:
with open("lines.txt", "w") as f:
    f.write("Line 1\nLine 2\nLine 3\n")

with open("lines.txt", "r") as f:
    for line in f:
        print(line, end="")

**Explanation:** Iterating a file object directly (for line in f) reads one line at a time lazily, which stays memory-efficient no matter how large the file is. Each line already includes its trailing \n, so end="" in print() avoids doubling up the line breaks.

## Exercise 4. Read File Lines into a List

**Concept:** f.readlines()

**Problem:** Load every line of a file into a Python list.

**Given:**
```
items.txt with apple/banana/cherry, one per line
```

**Expected Output:**
```
['apple\n', 'banana\n', 'cherry']
```

**Hint:** Each list element keeps its trailing \n except possibly the last line.

In [ ]:
with open("items.txt", "w") as f:
    f.write("apple\nbanana\ncherry")

with open("items.txt", "r") as f:
    lines = f.readlines()

print(lines)

**Explanation:** readlines() returns a list where each element is one line, still carrying its trailing \n newline character (the very last line only has one if the file itself ended with a newline). A comprehension like [line.strip() for line in lines] would give clean strings if the newlines aren't wanted.

## Exercise 5. Append New Sentence to Existing File

**Concept:** 'a' (append) mode vs. 'w' (write/overwrite)

**Problem:** Add a new sentence to a file without erasing what's already there.

**Given:**
```
notes.txt already containing "Original content."
```

**Expected Output:**
```
File now contains: Original content.\nThis is a new line.
```

**Hint:** 'a' mode positions the write cursor at the end of the file; 'w' would erase everything first.

In [ ]:
with open("notes.txt", "w") as f:
    f.write("Original content.")

with open("notes.txt", "a") as f:
    f.write("\nThis is a new line.")

print("Sentence appended successfully.")

with open("notes.txt") as f:
    print("Contents:", repr(f.read()))

**Explanation:** 'a' mode never erases existing content — new writes simply continue from the end of the file. The leading \n in the appended text ensures the new sentence starts on its own line rather than being joined directly onto the end of the existing text. Using 'w' here by mistake would silently destroy the original content, a common and easy-to-miss error.

## Exercise 6. Clear All File Content

**Concept:** opening in 'w' mode truncates immediately, even with no write() call

**Problem:** Empty an existing file completely.

**Given:**
```
temp.txt containing "Some old data."
```

**Expected Output:**
```
temp.txt exists but is 0 bytes
```

**Hint:** Simply opening a file in 'w' mode truncates it to zero length before any write() happens.

In [ ]:
with open("temp.txt", "w") as f:
    f.write("Some old data.")

with open("temp.txt", "w") as f:
    pass

print("File content cleared.")

import os
print("File size now:", os.path.getsize("temp.txt"), "bytes")

**Explanation:** The instant open("temp.txt", "w") runs, the file is truncated to zero bytes — this happens regardless of whether write() is ever called. pass is just a no-op placeholder that makes the with block syntactically valid; since nothing is written, the file stays empty.

## Exercise 7. Write Text to New File

**Concept:** f.writelines() with a list of pre-newlined strings

**Problem:** Write three separate lines to a new file.

**Given:**
```
no external input; three lines are predefined
```

**Expected Output:**
```
output.txt containing three lines: First line / Second line / Third line
```

**Hint:** writelines() does not add newlines automatically — each string in the list needs its own trailing \n.

In [ ]:
lines = ["First line\n", "Second line\n", "Third line\n"]

with open("output.txt", "w") as f:
    f.writelines(lines)

print("Text written to output.txt")

with open("output.txt") as f:
    print(f.read())

**Explanation:** Each string in the lines list already ends with \n, since writelines() writes them one after another with no separator of its own added. This is functionally equivalent to calling f.write() three times in a row, just more concise for a known list of lines.

## Exercise 8. Check If File Exists

**Concept:** os.path.exists()

**Problem:** Check whether a file exists before doing anything with it.

**Given:**
```
data.txt (created earlier in this notebook, so it exists)
```

**Expected Output:**
```
File exists.
```

**Hint:** os.path.exists() returns a boolean — True for either a file or a directory at that path.

In [ ]:
import os

if os.path.exists("data.txt"):
    print("File exists.")
else:
    print("File does not exist.")

if os.path.exists("this_file_does_not_exist.txt"):
    print("File exists.")
else:
    print("File does not exist.")

**Explanation:** os.path.exists() checks whether the given path points to an existing file or directory, returning True or False without raising an error either way. Checking this before opening a file in read mode is defensive programming — without it, attempting to open a missing file raises FileNotFoundError and crashes an unprotected script.

## Exercise 9. Handle Missing File with Try-Except

**Concept:** try/except FileNotFoundError

**Problem:** Attempt to open a file that doesn't exist, and fail gracefully.

**Given:**
```
missing.txt does not exist
```

**Expected Output:**
```
Error: The file was not found.
```

**Hint:** Catching the specific FileNotFoundError (not a bare except) lets other unrelated errors still surface normally.

In [ ]:
try:
    with open("missing.txt", "r") as f:
        content = f.read()
        print(content)
except FileNotFoundError:
    print("Error: The file was not found.")

**Explanation:** Python attempts everything inside the try block; the moment open() raises FileNotFoundError (because the file genuinely doesn't exist), execution jumps straight to the except clause instead of crashing the program. Naming the specific exception type means only this particular failure is caught — a different, unrelated error would still propagate and be visible.

## Exercise 10. Count Total Lines in File

**Concept:** len(f.readlines())

**Problem:** Count how many lines a file contains.

**Given:**
```
data.txt with 4 lines: Hello/World/Python/File Handling
```

**Expected Output:**
```
Total lines: 4
```

**Hint:** readlines() returns one list element per line; len() of that list is the line count.

In [ ]:
with open("data.txt", "w") as f:
    f.write("Hello\nWorld\nPython\nFile Handling\n")

with open("data.txt", "r") as f:
    lines = f.readlines()

print("Total lines:", len(lines))

**Explanation:** readlines() returns one string per line, so the length of that list directly equals the line count, regardless of whether the final line ends with a newline or not. For very large files, sum(1 for line in f) counts lazily without holding every line in memory at once.

## Exercise 11. Count Total Words in File

**Concept:** str.split() with no arguments splits on any whitespace

**Problem:** Count the total number of words across all lines of a file.

**Given:**
```
data.txt with 3 lines totaling 9 words
```

**Expected Output:**
```
Total words: 9
```

**Hint:** content.split() with no arguments splits on spaces, tabs, AND newlines all at once.

In [ ]:
with open("data.txt", "w") as f:
    f.write("Hello World\nPython is great\nFile handling is easy")

with open("data.txt", "r") as f:
    content = f.read()

words = content.split()
print("Total words:", len(words))

**Explanation:** Reading the whole file with .read() (rather than line by line) lets .split() work across every line boundary at once, since split() with no arguments treats spaces, tabs, and newlines all as equivalent separators. It also automatically collapses consecutive whitespace and ignores leading/trailing gaps, so len(words) gives an accurate word count.

## Exercise 12. Count Total Characters in File

**Concept:** len() on the full file string

**Problem:** Count every character in a file, including spaces and newlines.

**Given:**
```
data.txt containing "Hello, World!"
```

**Expected Output:**
```
Total characters: 13
```

**Hint:** len(content) counts Unicode characters (code points) in Python 3, not raw bytes.

In [ ]:
with open("data.txt", "w") as f:
    f.write("Hello, World!")

with open("data.txt", "r") as f:
    content = f.read()

print("Total characters:", len(content))

**Explanation:** len(content) counts every character in the string — letters, punctuation, spaces, and any embedded newlines — all treated as Unicode characters in Python 3, not raw bytes. To exclude newlines from the count, you could call len(content.replace("\n", "")) instead.

## Exercise 13. Count Specific Word Occurrences in File

**Concept:** str.count()

**Problem:** Count how many times a specific word appears in a file.

**Given:**
```
data.txt mentioning "Python" three times
```

**Expected Output:**
```
Occurrences of 'Python': 3
```

**Hint:** content.count() is case-sensitive by default and matches substrings, even inside larger words.

In [ ]:
word_to_find = "Python"

with open("data.txt", "w") as f:
    f.write("Python is great.\nI love Python.\nPython makes file handling easy.")

with open("data.txt", "r") as f:
    content = f.read()

count = content.count(word_to_find)
print(f"Occurrences of '{word_to_find}':", count)

**Explanation:** content.count(word) scans the entire file content string and tallies non-overlapping occurrences of that exact substring — case-sensitively by default, so 'python' or 'PYTHON' wouldn't count. Note .count() also matches inside larger words (searching 'on' would match part of 'Python' too); for true whole-word matching, split into tokens first or use the re module with word-boundary markers.

## Exercise 14. Read Only First N Lines

**Concept:** enumerate() + break to stop reading early

**Problem:** Print only the first 3 lines of a file, without reading the rest.

**Given:**
```
data.txt with 5 lines
```

**Expected Output:**
```
Line 1
Line 2
Line 3
```

**Hint:** Because the file object is lazy, break stops Python from ever reading the remaining lines from disk.

In [ ]:
with open("data.txt", "w") as f:
    f.write("Line 1\nLine 2\nLine 3\nLine 4\nLine 5\n")

n = 3

with open("data.txt", "r") as f:
    for i, line in enumerate(f):
        if i >= n:
            break
        print(line, end="")

**Explanation:** enumerate(f) pairs each line with an index starting at 0. The moment that index reaches n, break exits the loop immediately — and because the file object reads lazily, Python genuinely never reads the remaining lines from disk. This is far more efficient for large files than f.readlines()[:n], which would load the entire file into memory before slicing off just the first few lines.

## Exercise 15. Read Only Last N Lines

**Concept:** negative list slicing [-n:]

**Problem:** Print only the last 3 lines of a file.

**Given:**
```
data.txt with 5 lines
```

**Expected Output:**
```
Line 3
Line 4
Line 5
```

**Hint:** lines[-n:] safely returns all lines if n is larger than the total line count, with no error.

In [ ]:
with open("data.txt", "w") as f:
    f.write("Line 1\nLine 2\nLine 3\nLine 4\nLine 5\n")

n = 3

with open("data.txt", "r") as f:
    lines = f.readlines()

for line in lines[-n:]:
    print(line, end="")

**Explanation:** Getting the tail of a file requires knowing where the end is, which means readlines() must load every line first — there's no way around that with plain file reading. Python's negative slice lines[-n:] then grabs exactly the last n elements; for very large files, collections.deque(f, maxlen=n) achieves the same result while only ever holding n lines in memory as it streams through.

## Exercise 16. Read Specific Line Numbers from File

**Concept:** enumerate(f, start=1) + set membership

**Problem:** Print only specific line numbers (1, 3, and 5) from a file.

**Given:**
```
data.txt with 5 lines
```

**Expected Output:**
```
Line 1
Line 3
Line 5
```

**Hint:** A set (not a list) is used for target_lines since `in` membership checks are O(1) for sets.

In [ ]:
with open("data.txt", "w") as f:
    f.write("Line 1\nLine 2\nLine 3\nLine 4\nLine 5\n")

target_lines = {1, 3, 5}

with open("data.txt", "r") as f:
    for line_num, line in enumerate(f, start=1):
        if line_num in target_lines:
            print(line, end="")

**Explanation:** enumerate(f, start=1) numbers lines starting at 1, matching natural human line-counting rather than Python's usual 0-based indexing. Checking line_num in target_lines (a set) is a fast, clear way to select only the desired lines; for a very large file with all targets near the top, adding `if line_num > max(target_lines): break` would let you stop reading early.

## Exercise 17. Find Longest Word in File

**Concept:** tracking a running 'longest so far' value

**Problem:** Find the longest word in a text file.

**Given:**
```
words.txt containing "Python is a powerful programming language"
```

**Expected Output:**
```
Longest word: programming
```

**Hint:** max(words, key=len) is the concise one-line alternative to the explicit tracking loop.

In [ ]:
with open("words.txt", "w") as f:
    f.write("Python is a powerful programming language")

with open("words.txt", "r") as f:
    words = f.read().split()

longest = ""
for word in words:
    if len(word) > len(longest):
        longest = word

print("Longest word:", longest)

**Explanation:** Starting longest as an empty string gives every real word a chance to beat it on the first comparison. Each iteration only replaces longest when a strictly longer word is found, so the final value is guaranteed to be the single longest word in the whole list; max(words, key=len) would achieve the identical result in one line.

## Exercise 18. Count Each Letter Frequency in File

**Concept:** dict.get() as a safe counter, filtered by str.isalpha()

**Problem:** Count how often each letter appears in a file, ignoring case and non-letters.

**Given:**
```
sample.txt containing "Hello World"
```

**Expected Output:**
```
h: 1, e: 1, l: 3, o: 2, w: 1, r: 1, d: 1
```

**Hint:** .isalpha() filters out spaces and punctuation before counting; .lower() makes the count case-insensitive.

In [ ]:
with open("sample.txt", "w") as f:
    f.write("Hello World")

freq = {}

with open("sample.txt", "r") as f:
    content = f.read().lower()

for char in content:
    if char.isalpha():
        freq[char] = freq.get(char, 0) + 1

for letter, count in freq.items():
    print(f"{letter}: {count}")

**Explanation:** .lower() first ensures 'H' and 'h' are counted as the same letter. Looping character by character, .isalpha() filters out the space (and would filter punctuation too), so only actual letters get tallied. freq.get(char, 0) + 1 safely increments a count that may not exist yet, defaulting to 0 for a letter's first appearance.

## Exercise 19. Search Word and Print Matching Line Numbers

**Concept:** combining enumerate() with the `in` membership operator

**Problem:** Find every line number where a specific word appears.

**Given:**
```
notes.txt where "Python" appears on lines 1 and 3
```

**Expected Output:**
```
"Python" found on line 1
"Python" found on line 3
```

**Hint:** Lowercasing both the search word and the line makes the match case-insensitive.

In [ ]:
with open("notes.txt", "w") as f:
    f.write("Python is great.\nI enjoy coding.\nPython makes it easy.\n")

search_word = "Python"

with open("notes.txt", "r") as f:
    for line_num, line in enumerate(f, start=1):
        if search_word.lower() in line.lower():
            print(f'"{search_word}" found on line {line_num}')

**Explanation:** enumerate(f, start=1) gives both the 1-based line number and the line's text on every iteration. Lowercasing both search_word and line before the `in` check means the match succeeds regardless of the original capitalization in either the search term or the file's text.

## Exercise 20. Strip Extra Whitespace and Save to New File

**Concept:** reading and writing two files at once, .strip() per line

**Problem:** Remove leading/trailing whitespace from every line and save the cleaned result to a new file.

**Given:**
```
messy.txt with lines like "  Hello World  " and "   Python   "
```

**Expected Output:**
```
clean.txt containing 'Hello World' and 'Python', with no surrounding whitespace
```

**Hint:** A single with statement can open both the input and output files together, comma-separated.

In [ ]:
with open("messy.txt", "w") as f:
    f.write("  Hello World  \n   Python   \n")

with open("messy.txt", "r") as infile, open("clean.txt", "w") as outfile:
    for line in infile:
        cleaned_line = line.strip()
        outfile.write(cleaned_line + "\n")

with open("clean.txt") as f:
    print(repr(f.read()))

**Explanation:** Opening both files in one with statement (comma-separated) guarantees both close properly once the block ends. .strip() removes all leading and trailing whitespace, including the original line's trailing \n — which is exactly why + "\n" has to be added back manually when writing each cleaned line to the output file.

## Exercise 21. Convert Uppercase to Lowercase and Vice Versa

**Concept:** str.swapcase()

**Problem:** Swap the case of every letter in a file and save the result.

**Given:**
```
input.txt containing "Hello World"
```

**Expected Output:**
```
swapped.txt containing 'hELLO wORLD'
```

**Hint:** swapcase() toggles every letter at once; non-letter characters are left untouched.

In [ ]:
with open("input.txt", "w") as f:
    f.write("Hello World")

with open("input.txt", "r") as infile:
    content = infile.read()

swapped = content.swapcase()

with open("swapped.txt", "w") as outfile:
    outfile.write(swapped)

with open("swapped.txt") as f:
    print(f.read())

**Explanation:** swapcase() returns a brand-new string where every uppercase letter becomes lowercase and every lowercase letter becomes uppercase in a single call, leaving spaces and punctuation unchanged. Since the original newlines are already preserved inside the read string, writing swapped directly needs no extra manual newline handling.

## Exercise 22. Find and Replace a Word Throughout File

**Concept:** read-modify-write: two separate with blocks for the same file

**Problem:** Replace every occurrence of one word with another throughout a file.

**Given:**
```
story.txt containing "I love Java. Java is great."
```

**Expected Output:**
```
story.txt updated to: I love Python. Python is great.
```

**Hint:** Read and close the file first, THEN reopen it in write mode — don't try to read and write in the same session.

In [ ]:
with open("story.txt", "w") as f:
    f.write("I love Java. Java is great.")

old_word = "Java"
new_word = "Python"

with open("story.txt", "r") as f:
    content = f.read()

updated_content = content.replace(old_word, new_word)

with open("story.txt", "w") as f:
    f.write(updated_content)

with open("story.txt") as f:
    print(f.read())

**Explanation:** content.replace(old_word, new_word) is case-sensitive and substitutes every occurrence throughout the whole string at once. The read and write happen in two deliberately separate with blocks — reading everything into memory first, fully closing that file handle, and only then reopening the same path in "w" mode (which truncates it) to write the updated content back; attempting to read and write the same file within a single open session can risk data loss on some systems.

## Exercise 23. Get File Size in Kilobytes

**Concept:** os.path.getsize()

**Problem:** Find a file's size in kilobytes.

**Given:**
```
data.txt sized at some number of bytes
```

**Expected Output:**
```
File size: X.XX KB
```

**Hint:** os.path.getsize() returns bytes; divide by 1024 to convert to kilobytes.

In [ ]:
import os

with open("data.txt", "w") as f:
    f.write("x" * 2048)

filename = "data.txt"
size_bytes = os.path.getsize(filename)
size_kb = size_bytes / 1024

print(f"File size: {size_kb} KB")

**Explanation:** os.path.getsize() returns a file's size directly in bytes without needing to open or read it. Dividing by 1024 converts to kilobytes, and since Python 3's / always performs true division, the result comes back as a precise float (2.0 here) even for sizes that aren't perfectly round.

## Exercise 24. Copy File Using Binary Mode

**Concept:** 'rb'/'wb' for an exact, format-agnostic byte-for-byte copy

**Problem:** Copy a file using binary mode so it works correctly for any file type, not just text.

**Given:**
```
source.txt (standing in for any file type)
```

**Expected Output:**
```
File copied successfully.
```

**Hint:** Binary mode ('rb'/'wb') applies no character-encoding conversion, making the copy exact regardless of content.

In [ ]:
with open("source.txt", "w") as f:
    f.write("This could be any file type - text, image, PDF, etc.")

with open("source.txt", "rb") as src, open("destination.txt", "wb") as dst:
    data = src.read()
    dst.write(data)

print("File copied successfully.")

with open("destination.txt", "rb") as f:
    print(f.read())

**Explanation:** The 'b' suffix in "rb"/"wb" tells Python to treat the file as raw bytes rather than decoded text, so no newline translation or character-encoding conversion happens at all. This is what makes the copy exact for any file type — images, PDFs, executables — not just plain text. For very large files, reading in a fixed-size loop (like Exercise 31) is more memory-efficient than loading everything with one .read() call.

## Exercise 25. Rename a Single File

**Concept:** os.rename()

**Problem:** Rename a file on disk using the os module.

**Given:**
```
old_name.txt exists on disk
```

**Expected Output:**
```
File renamed successfully.
```

**Hint:** Wrapping the call in try/except FileNotFoundError handles the case where the source file is missing.

In [ ]:
import os

with open("old_name.txt", "w") as f:
    f.write("some content")

old_name = "old_name.txt"
new_name = "new_name.txt"

try:
    os.rename(old_name, new_name)
    print("File renamed successfully.")
except FileNotFoundError:
    print(f"Error: '{old_name}' not found.")

print("Exists now:", os.path.exists(new_name))

**Explanation:** os.rename(old, new) renames the file at the first path to the second — on most systems this is an atomic operation as long as both paths are on the same filesystem. Wrapping the call in try/except FileNotFoundError prevents a crash if the source file has already been moved or never existed, giving a clear message instead. Note os.rename() can also move a file into a different directory if the new path includes one.

## Exercise 26. Rename Multiple Files with Prefix

**Concept:** os.listdir() + filtering + os.rename() in a loop

**Problem:** Add a prefix to every .txt file in a directory.

**Given:**
```
a sandboxed folder containing notes.txt and data.txt
```

**Expected Output:**
```
Renamed: notes.txt -> 2024_notes.txt
Renamed: data.txt -> 2024_data.txt
```

**Hint:** os.path.join() builds correct, portable paths for any operating system.

In [ ]:
import os

directory = "rename_demo"
os.makedirs(directory, exist_ok=True)
for name in ["notes.txt", "data.txt"]:
    with open(os.path.join(directory, name), "w") as f:
        f.write("sample content")

prefix = "2024_"

for filename in sorted(os.listdir(directory)):
    if filename.endswith(".txt") and not filename.startswith(prefix):
        old_path = os.path.join(directory, filename)
        new_path = os.path.join(directory, prefix + filename)
        os.rename(old_path, new_path)
        print(f"Renamed: {filename} -> {prefix + filename}")

**Explanation:** os.listdir(directory) lists every entry in the folder; filtering with filename.endswith(".txt") targets only text files, leaving anything else untouched. os.path.join(directory, filename) builds a path using the correct separator for the current operating system, which is essential for os.rename() to locate the file correctly regardless of whether the code runs on Windows or Unix. (This demo uses a small sandboxed 'rename_demo' folder rather than the notebook's working directory.)

## Exercise 27. Delete a File from Disk

**Concept:** os.path.exists() guard + os.remove()

**Problem:** Safely delete a file, checking first that it actually exists.

**Given:**
```
temp.txt exists on disk
```

**Expected Output:**
```
temp.txt has been deleted.
```

**Hint:** os.remove() permanently deletes the file — there's no recycle bin involved.

In [ ]:
import os

with open("temp.txt", "w") as f:
    f.write("temporary content")

filename = "temp.txt"

if os.path.exists(filename):
    os.remove(filename)
    print(f"{filename} has been deleted.")
else:
    print(f"{filename} does not exist.")

print("Still exists?", os.path.exists(filename))

**Explanation:** Checking os.path.exists() first guards against a FileNotFoundError that os.remove() would otherwise raise on a missing file. os.remove() permanently unlinks the file from the filesystem with no recycle bin or undo — which is exactly why the existence check (and generally, caution) matters before calling it. Note os.remove() only works on files; deleting an empty directory needs os.rmdir(), and a non-empty one needs shutil.rmtree().

## Exercise 28. Merge Two Files into One

**Concept:** looping over multiple input files, writing to one shared output handle

**Problem:** Combine the contents of two files into a single new file.

**Given:**
```
file1.txt: "Hello from file 1.", file2.txt: "Hello from file 2."
```

**Expected Output:**
```
merged.txt containing both lines; Files merged into merged.txt
```

**Hint:** Opening the output file once, outside the loop, keeps every source file's content in the same write session.

In [ ]:
with open("file1.txt", "w") as f:
    f.write("Hello from file 1.")
with open("file2.txt", "w") as f:
    f.write("Hello from file 2.")

input_files = ["file1.txt", "file2.txt"]
output_file = "merged.txt"

with open(output_file, "w") as outfile:
    for filename in input_files:
        with open(filename, "r") as infile:
            outfile.write(infile.read())
            outfile.write("\n")

print(f"Files merged into {output_file}")

with open(output_file) as f:
    print(f.read())

**Explanation:** Storing the source filenames in a list makes it trivial to merge more files later just by adding more entries. Opening output_file once, before the loop begins, means every source file's content gets appended into the same open write session rather than being reopened (and truncated) repeatedly. The extra outfile.write("\n") after each file's content prevents the last line of one file from running directly into the first line of the next.

## Exercise 29. Reverse Line Order and Save to New File

**Concept:** list slicing [::-1] on the lines list

**Problem:** Reverse the order of a file's lines and save the result to a new file.

**Given:**
```
original.txt with 3 lines
```

**Expected Output:**
```
reversed.txt with lines in reverse order: Line 3 / Line 2 / Line 1
```

**Hint:** [::-1] creates a reversed copy of the list without modifying the original.

In [ ]:
with open("original.txt", "w") as f:
    f.write("Line 1\nLine 2\nLine 3\n")

with open("original.txt", "r") as f:
    lines = f.readlines()

reversed_lines = lines[::-1]

with open("reversed.txt", "w") as f:
    f.writelines(reversed_lines)

print("Lines reversed and saved to reversed.txt")

with open("reversed.txt") as f:
    print(f.read())

**Explanation:** readlines() loads every line (each still carrying its trailing \n) into a list, which is necessary to be able to reorder them at all. lines[::-1] builds a new, reversed copy of that list without touching the original. writelines() then writes each string in that reversed list in sequence — since every line already contains its own newline character, no extra separator is needed between them.

## Exercise 30. List All Files in Directory and Save

**Concept:** os.listdir() + os.path.isfile() to exclude subdirectories

**Problem:** List only the files (not subfolders) in a directory and save the names to a text file.

**Given:**
```
a sandboxed folder containing notes.txt, data.csv, report.pdf, and one subfolder
```

**Expected Output:**
```
file_list.txt listing just the three filenames, one per line
```

**Hint:** os.path.isfile() filters out subdirectories that os.listdir() would otherwise also include.

In [ ]:
import os

directory = "my_folder"
os.makedirs(directory, exist_ok=True)
os.makedirs(os.path.join(directory, "a_subfolder"), exist_ok=True)
for name in ["notes.txt", "data.csv", "report.pdf"]:
    with open(os.path.join(directory, name), "w") as f:
        f.write("sample content")

output_file = "file_list.txt"

with open(output_file, "w") as outfile:
    for entry in sorted(os.listdir(directory)):
        full_path = os.path.join(directory, entry)
        if os.path.isfile(full_path):
            outfile.write(entry + "\n")

print(f"File list saved to {output_file}")

with open(output_file) as f:
    print(f.read())

**Explanation:** os.listdir(directory) returns every entry in the folder — both files and subdirectories — in an order that isn't guaranteed and varies by OS (sorted() here keeps the output predictable). os.path.join(directory, entry) builds the full path needed for os.path.isfile() to correctly check each entry regardless of the current working directory; that check is what excludes 'a_subfolder' from the final list, keeping only the three genuine files.

## Exercise 31. Read and Write Binary Image File

**Concept:** chunked binary copy — reading fixed-size pieces in a loop

**Problem:** Copy a binary file (like an image) in fixed-size chunks rather than all at once.

**Given:**
```
photo.jpg (simulated with placeholder bytes for this demo)
```

**Expected Output:**
```
Image copied to photo_copy.jpg
```

**Hint:** An empty bytes object (b'') from read() is falsy, which is exactly what signals the end of the file.

In [ ]:
with open("photo.jpg", "wb") as f:
    f.write(b"\xff\xd8\xff" + b"FAKE_JPEG_DATA_FOR_DEMO" * 500)

source = "photo.jpg"
destination = "photo_copy.jpg"
chunk_size = 4096

with open(source, "rb") as src, open(destination, "wb") as dst:
    while True:
        chunk = src.read(chunk_size)
        if not chunk:
            break
        dst.write(chunk)

print(f"Image copied to {destination}")

import os
print("Sizes match:", os.path.getsize(source) == os.path.getsize(destination))

**Explanation:** "rb"/"wb" mode reads and writes raw bytes with zero newline translation or encoding conversion — essential for a binary format like JPEG, where any text-mode 'helpfulness' would corrupt the file. src.read(chunk_size) pulls only up to 4096 bytes at a time instead of the whole file, keeping memory usage flat and constant even for very large files. When the source is exhausted, read() returns an empty bytes object (b""), which evaluates as falsy — that's exactly what if not chunk: break detects to end the loop cleanly.

## Exercise 32. Extract and Sort Unique Words from File

**Concept:** punctuation stripping + set() for dedup + sorted()

**Problem:** Extract every unique word from a file, ignoring punctuation and case, sorted alphabetically.

**Given:**
```
paragraph.txt containing "the cat sat on the mat the cat"
```

**Expected Output:**
```
cat, mat, on, sat, the (each printed once, alphabetically)
```

**Hint:** str.maketrans('', '', string.punctuation) builds a translation table that deletes every punctuation character.

In [ ]:
import string

with open("paragraph.txt", "w") as f:
    f.write("the cat sat on the mat the cat")

with open("paragraph.txt", "r") as f:
    content = f.read().lower()

words = content.translate(str.maketrans("", "", string.punctuation)).split()
unique_sorted = sorted(set(words))

for word in unique_sorted:
    print(word)

**Explanation:** .lower() first normalizes case so 'The' and 'the' are treated as one word. str.maketrans("", "", string.punctuation) builds a translation table mapping every punctuation character to nothing at all, so .translate() strips commas, periods, and similar symbols before .split() breaks the text into words. Converting that word list to a set() discards duplicates automatically, and sorted() then produces a stable, predictable alphabetical ordering (sets themselves have no guaranteed order).

## Exercise 33. Filter Log File Lines Containing ERROR Keyword

**Concept:** the `in` operator for line-by-line keyword filtering

**Problem:** Print only the lines of a log file that contain a specific keyword.

**Given:**
```
app.log with a mix of INFO and ERROR lines
```

**Expected Output:**
```
Only the two ERROR lines are printed
```

**Hint:** Iterating the file object one line at a time keeps memory use low, important for large log files.

In [ ]:
with open("app.log", "w") as f:
    f.write(
        "2024-01-01 10:00:00 INFO Server started\n"
        "2024-01-01 10:02:05 ERROR Disk quota exceeded\n"
        "2024-01-01 10:05:00 INFO Request processed\n"
        "2024-01-01 10:07:45 ERROR Connection timeout\n"
        "2024-01-01 10:09:00 INFO Shutdown complete\n"
    )

keyword = "ERROR"

with open("app.log", "r") as f:
    for line in f:
        if keyword in line:
            print(line.strip())

**Explanation:** Iterating the file object directly (for line in f) reads one line at a time on demand, which matters for real log files that can grow to gigabytes. The `in` operator performs a simple, case-sensitive substring check on each line — 'error' or 'Error' would not match this particular search. .strip() removes the trailing newline before printing, avoiding blank-looking gaps between each matched line in the output.

## Exercise 34. Split Large File into Smaller 10-Line Files

**Concept:** chunked slicing with range(0, total, chunk_size)

**Problem:** Split a large text file into smaller files of at most 10 lines each.

**Given:**
```
large_file.txt with 25 lines
```

**Expected Output:**
```
part_1.txt (10 lines), part_2.txt (10 lines), part_3.txt (5 lines)
```

**Hint:** Slicing past the end of a list (like the final, shorter chunk) is always safe in Python and never raises an error.

In [ ]:
with open("large_file.txt", "w") as f:
    for i in range(1, 26):
        f.write(f"This is line number {i}\n")

chunk_size = 10

with open("large_file.txt", "r") as f:
    lines = f.readlines()

total_lines = len(lines)

for i, start in enumerate(range(0, total_lines, chunk_size), start=1):
    chunk = lines[start:start + chunk_size]
    output_filename = f"part_{i}.txt"
    with open(output_filename, "w") as out:
        out.writelines(chunk)
    print(f"Created: {output_filename} ({len(chunk)} lines)")

**Explanation:** range(0, total_lines, chunk_size) generates starting indices 10 apart (0, 10, 20, ...), letting the loop step through the full line list in non-overlapping chunks. lines[start:start + chunk_size] slices out exactly that many lines each time — for the final, shorter chunk, Python's slicing safely returns however many lines remain instead of raising an error. enumerate(..., start=1) numbers each chunk starting at 1, which is used directly to build filenames like part_1.txt, part_2.txt, and so on.